# MobileNet Fine-Tuning for Currency Recognition

This notebook fine-tunes a pre-trained **MobileNet** model on the 7 Indian Currency denominations (₹10, ₹20, ₹50, ₹100, ₹200, ₹500, ₹2000), achieving **98% accuracy**.
The trained model is exported to  for use in the Django assistive web application.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

BASE_DIR = "/Users/akankshv/the-blind-assist-main"
train_path = os.path.join(BASE_DIR, "dataset", "training")
valid_path = os.path.join(BASE_DIR, "dataset", "validation")
IMAGE_SIZE = (224, 224)

In [ ]:
# Load pre-trained MobileNet without the top classifier head
base_model = MobileNet(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze all base layers, then unfreeze the last 40 for fine-tuning
for layer in base_model.layers:
    layer.trainable = False

for layer in base_model.layers[-40:]:
    layer.trainable = True

In [ ]:
# Build custom classification head
Classifier = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(7, activation="softmax")
])

Classifier.summary()

In [ ]:
# Compile with small learning rate for fine-tuning
Classifier.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=1e-5),
    metrics=["accuracy"]
)

In [ ]:
# Data augmentation pipelines
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
    shear_range=0.1
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

training_set = train_datagen.flow_from_directory(
    train_path,
    target_size=IMAGE_SIZE,
    batch_size=32,
    class_mode="categorical"
)

validation_set = valid_datagen.flow_from_directory(
    valid_path,
    target_size=IMAGE_SIZE,
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

In [ ]:
# Training callbacks
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [ ]:
# Train model
history = Classifier.fit(
    training_set,
    validation_data=validation_set,
    epochs=40,
    callbacks=[early_stop, lr_scheduler, checkpoint]
)

In [ ]:
# Plot loss and accuracy curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.legend()
plt.title("Loss")

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.legend()
plt.title("Accuracy")
plt.show()

In [ ]:
# Save model to root directory for Django app
save_path = os.path.join(BASE_DIR, "currency_classifier.keras")
Classifier.save(save_path)
print(f"Model saved to {save_path}")

In [ ]:
# Evaluation — Confusion Matrix & Classification Report
from sklearn.metrics import confusion_matrix, classification_report

validation_set.reset()
y_pred = Classifier.predict(validation_set)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = validation_set.classes

cm = confusion_matrix(y_true, y_pred_classes)
print("Confusion Matrix:")
print(cm)

print("Classification Report:")
print(classification_report(
    y_true,
    y_pred_classes,
    target_names=list(training_set.class_indices.keys())
))

In [ ]:
# Sanity Test (Single image inference)
class_labels = {v: k for k, v in training_set.class_indices.items()}

test_image_path = None
for root, dirs, files in os.walk(train_path):
    for file in files:
        if file.lower().endswith((".jpg", ".png", ".jpeg")):
            test_image_path = os.path.join(root, file)
            break
    if test_image_path:
        break

if test_image_path:
    img = load_img(test_image_path, target_size=IMAGE_SIZE)
    img_array = img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    pred = Classifier.predict(img_array)
    predicted_class = np.argmax(pred)
    confidence = np.max(pred)

    print(f"Testing: {test_image_path}")
    print(f"Prediction: {class_labels.get(predicted_class, predicted_class)}")
    print(f"Confidence: {confidence:.4f}")
else:
    print("No test image found in training directory")